# 04 Feature Engineering

This notebook creates lag and rolling PM2.5 features.

Note:
- these lag features use previous PM2.5 history
- this usually improves R2 a lot compared with weather-only prediction

In [6]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

BASE_DIR = Path.cwd()
AQI_DIR = BASE_DIR / "data" / "AQI"
HTML_DIR = BASE_DIR / "data" / "html_data"
ARTIFACTS_DIR = BASE_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [7]:
data = joblib.load(ARTIFACTS_DIR / "pm25_base_model_data.joblib")
data.head()

,Date,Avg_Temperature,Max_Temperature,Min_Temperature,Humidity,Rainfall_Snowmelt,Visibility,Wind_Speed,Max_Sustained_Wind_Speed,City,Month,Year,Day,DayOfWeek,IsWeekend,Season,Daily_PM25
0,2013-01-01,23.4,30.3,19.0,59.0,0.0,6.3,4.3,5.4,Bangalore,1,2013,1,1,0,Winter,279.156522
1,2013-01-02,22.4,30.3,16.9,57.0,0.0,6.9,3.3,7.6,Bangalore,1,2013,2,2,0,Winter,332.239130
2,2013-01-03,24.0,31.8,16.9,51.0,0.0,6.9,2.8,5.4,Bangalore,1,2013,3,3,0,Winter,38.846154
3,2013-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Bangalore,1,2013,4,4,0,Winter,46.416667
4,2013-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Bangalore,1,2013,5,5,1,Winter,92.391304


In [8]:
feature_data = data.copy()

for lag in [1, 2, 3, 5, 7, 10, 14, 21, 30]:
    feature_data[f"PM25_lag_{lag}"] = feature_data["Daily_PM25"].shift(lag)

for window in [3, 5, 7, 10, 14, 21, 30]:
    feature_data[f"PM25_roll_mean_{window}"] = feature_data["Daily_PM25"].shift(1).rolling(window).mean()

feature_data["PM25_change_1"] = feature_data["Daily_PM25"].shift(1) - feature_data["Daily_PM25"].shift(2)
feature_data["PM25_change_3"] = feature_data["Daily_PM25"].shift(1) - feature_data["Daily_PM25"].shift(4)
feature_data["Temp_Range"] = feature_data["Max_Temperature"] - feature_data["Min_Temperature"]
feature_data["Month_sin"] = np.sin(2 * np.pi * feature_data["Month"] / 12)
feature_data["Month_cos"] = np.cos(2 * np.pi * feature_data["Month"] / 12)

feature_data = feature_data.dropna().reset_index(drop=True)

print("Feature engineered dataset shape:", feature_data.shape)
feature_data.head()

Feature engineered dataset shape: (77, 38)


,Date,Avg_Temperature,Max_Temperature,Min_Temperature,Humidity,Rainfall_Snowmelt,Visibility,Wind_Speed,Max_Sustained_Wind_Speed,City,Month,Year,Day,DayOfWeek,IsWeekend,Season,Daily_PM25,PM25_lag_1,PM25_lag_2,PM25_lag_3,PM25_lag_5,PM25_lag_7,PM25_lag_10,PM25_lag_14,PM25_lag_21,PM25_lag_30,PM25_roll_mean_3,PM25_roll_mean_5,PM25_roll_mean_7,PM25_roll_mean_10,PM25_roll_mean_14,PM25_roll_mean_21,PM25_roll_mean_30,PM25_change_1,PM25_change_3,Temp_Range,Month_sin,Month_cos
0,2014-02-07,23.8,32.6,16.5,29.0,0.0,6.9,2.4,5.4,Bangalore,2,2014,7,4,0,Winter,104.222222,75.000000,92.478261,107.217391,239.826087,68.133333,19.869565,130.608696,59.130435,219.300000,91.565217,126.147826,132.014248,122.273289,118.388360,134.111909,123.553139,-17.478261,-41.217391,16.1,0.866025,5.000000e-01
1,2014-02-08,23.6,32.1,15.5,35.0,0.0,6.9,2.0,5.4,Bangalore,2,2014,8,5,1,Winter,45.869565,104.222222,75.000000,92.478261,116.217391,225.227273,62.937500,44.260870,169.000000,149.347826,90.566828,99.027053,137.169804,130.708555,116.503612,136.259137,119.717214,29.222222,-2.995169,16.6,0.866025,5.000000e-01
2,2014-09-07,23.3,28.7,20.3,77.0,0.0,6.9,6.3,9.4,Bangalore,9,2014,7,6,1,Monsoon,69.217391,85.818182,118.913043,67.260870,102.045455,106.043478,42.043478,205.000000,89.454545,297.130435,90.664032,103.477075,148.295031,133.371739,122.150473,129.059849,123.659615,-33.094862,-57.529644,8.4,-1.000000,-1.836970e-16
3,2014-09-08,23.2,28.9,19.8,71.0,0.0,6.3,9.1,11.1,Bangalore,9,2014,8,0,0,Monsoon,29.347826,69.217391,85.818182,118.913043,143.347826,414.636364,71.956522,76.391304,234.565217,100.954545,91.316206,96.911462,143.034161,136.089130,112.451715,128.096175,116.062513,-16.600791,1.956522,9.1,-1.000000,-1.836970e-16
4,2014-09-09,24.0,29.9,19.1,69.0,0.0,6.9,7.8,11.1,Bangalore,9,2014,9,1,0,Monsoon,50.086957,29.347826,69.217391,85.818182,67.260870,102.045455,181.652174,64.476190,118.086957,129.739130,61.461133,74.111462,87.992942,131.828261,109.091467,118.323918,113.675623,-39.869565,-89.565217,10.8,-1.000000,-1.836970e-16


In [9]:
feature_columns = [col for col in feature_data.columns if col not in ["Date", "Daily_PM25"]]
target_column = "Daily_PM25"

split_index = int(len(feature_data) * 0.75)
train_df = feature_data.iloc[:split_index].copy()
test_df = feature_data.iloc[split_index:].copy()

X_train = train_df[feature_columns]
X_test = test_df[feature_columns]
y_train = train_df[target_column]
y_test = test_df[target_column]

numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = [col for col in X_train.columns if col not in numeric_features]

preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_features),
    ("cat", Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Train rows:", len(train_df))
print("Test rows :", len(test_df))
print("Processed X_train shape:", X_train_processed.shape)
print("Processed X_test shape :", X_test_processed.shape)

Train rows: 57
Test rows : 20
Processed X_train shape: (57, 39)
Processed X_test shape : (20, 39)


In [10]:
joblib.dump(preprocessor, ARTIFACTS_DIR / "pm25_preprocessor.joblib")
joblib.dump(feature_data, ARTIFACTS_DIR / "pm25_feature_data.joblib")
joblib.dump(X_train_processed, ARTIFACTS_DIR / "pm25_X_train_processed.joblib")
joblib.dump(X_test_processed, ARTIFACTS_DIR / "pm25_X_test_processed.joblib")
joblib.dump(y_train.reset_index(drop=True), ARTIFACTS_DIR / "pm25_y_train.joblib")
joblib.dump(y_test.reset_index(drop=True), ARTIFACTS_DIR / "pm25_y_test.joblib")

print("Saved feature engineered artifacts.")

Saved feature engineered artifacts.
